In [60]:
import numpy as np
import pandas as pd

from sklearn.metrics import f1_score
from sklearn.ensemble import VotingClassifier
from scipy.optimize import minimize

In [61]:
# load logits of labels=1
ytrain = pd.read_csv("data/train.csv")["target"].to_numpy()

bert_train_logits = np.load("saved_var/oof_labels_prob_train.npy") # (-1, 1)
bert_test_logits = np.load("saved_var/oof_labels_prob_test.npy") # (-1, fold=5)

lgbm_train_logits = np.load("saved_var/lgbm/oof_train_logit.npy") # (-1, )
lgbm_test_logits = np.load("saved_var/lgbm/oof_test_logit.npy") # (fold=5,-1)

In [62]:
# averaging fold
bert_test_logits = bert_test_logits.mean(axis=1)
lgbm_test_logits = lgbm_test_logits.mean(axis=0)

In [63]:
# combining data
bert_train_logits = bert_train_logits.reshape(-1, 1)
bert_test_logits = bert_test_logits.reshape(-1, 1)
lgbm_train_logits = lgbm_train_logits.reshape(-1, 1)
lgbm_test_logits = lgbm_test_logits.reshape(-1, 1)

xtrain = np.concatenate([bert_train_logits, lgbm_train_logits], axis=1)
xtest = np.concatenate([bert_test_logits, lgbm_test_logits], axis=1)

In [64]:
def objective(weights):
    weights = np.clip(weights, 0, None)  # no negative weights
    weights /= np.sum(weights)
    blended = np.dot(xtrain, weights)
    blended_out = np.where(blended>0.4, 1, 0)
    return -f1_score(blended_out, ytrain)  # negative for minimization

# initialize equal weights
init = np.ones(xtrain.shape[1]) / xtrain.shape[1]

res = minimize(objective, init, method='Nelder-Mead')
opt_weights = res.x / np.sum(res.x)

print("Optimal weights:", opt_weights)
print("Best f1_score:", -res.fun)

Optimal weights: [0.95720165 0.04279835]
Best f1_score: 0.7946936197094125


In [65]:
prob_test = np.average(xtest, axis=1, weights=opt_weights)
labels_test = np.where(prob_test>0.4, 1, 0)

In [66]:
# submission
submit = pd.read_csv("data/sample_submission.csv")
submit["target"] = labels_test
submit.to_csv("data/submission.csv", index=False)